# Deployment: score the stream, end to end

The full path, live: deploy the endpoint → start the Flink scorer → replay
ticks into Kafka → read predictions off the output topic.

## Deploy the endpoint

In [ ]:
import boto3
from sagemaker.local import LocalSession
from sagemaker.model import Model

session = LocalSession()
BUCKET = "electricity-price-streaming"
s3 = boto3.client(
    "s3", endpoint_url="http://electricity-price-streaming-moto:5000"
)
jobs = sorted(
    o["Key"]
    for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="output/").get(
        "Contents", []
    )
    if o["Key"].endswith("model.tar.gz")
)
model = Model(
    image_uri="electricity-price-streaming:inference",
    model_data=f"s3://{BUCKET}/{jobs[-1]}",
    role="arn:aws:iam::111111111111:role/service-role/SageMakerRole",
    sagemaker_session=session,
)
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="local",
    endpoint_name="electricity-price-direction",
)
predictor.endpoint_name

## Create the topics

In [ ]:
%%bash
docker exec electricity-price-streaming-kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --create --if-not-exists --topic market-ticks --partitions 1 --replication-factor 1

In [ ]:
%%bash
docker exec electricity-price-streaming-kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --create --if-not-exists --topic price-predictions --partitions 1 --replication-factor 1

In [ ]:
%%bash
docker exec electricity-price-streaming-kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --list

## The Flink job

`flink_job/enrich_and_score.py` — for each tick: tick features →
feature vector → endpoint invoke → write to `price-predictions`.

In [ ]:
%%bash
cat flink_job/enrich_and_score.py

## Start the Flink stack + submit

In [ ]:
%%bash
make up-flink-jobmanager up-flink-taskmanager

In [ ]:
%%bash
docker exec electricity-price-streaming-flink-jm flink run -py /opt/flink_job/enrich_and_score.py || echo 'submit via jobmanager UI/CLI'

## Feed the market

In [ ]:
%%bash
head -3 data/processed/stream_events.jsonl

In [ ]:
import subprocess

# replay 500 ticks through the producer (runs in the notebook kernel —
# kafka-python talks to the broker over mlops_net)

proc = subprocess.Popen(
    [
        "python",
        "flink_job/kafka_producer.py",
        "--events",
        "data/processed/stream_events.jsonl",
        "--bootstrap",
        "electricity-price-streaming-kafka:9092",
        "--limit",
        "500",
        "--rate",
        "200",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
proc

## Read the predictions

In [ ]:
%%bash
docker exec electricity-price-streaming-kafka /opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server kafka:9092 --topic price-predictions --from-beginning --max-messages 5 --timeout-ms 15000 || true

Each output record is `{market_tick_id, prediction}` — tick in, scored
prediction out, decoupled by the topics.